In [148]:
import gmsh
import os
import time
import numpy as np
from dolfinx.io import gmshio
from mpi4py import MPI
import pyvista as pv
from dolfinx.plot import vtk_mesh
import cv2
import matplotlib.pyplot as plt

In [149]:
def remove_duplicate_params(parametricCoords, coords):
    parametricCoords = np.array(parametricCoords)
    coords = np.array(coords)
    
    _, unique_indices = np.unique(parametricCoords, return_index=True)
    
    return parametricCoords[unique_indices], coords[unique_indices]

In [150]:
def stitch_curves(curves, tol=1e-10):
    """
    Stitch curves given as lists of coordinate lists, based on shared endpoints.

    Parameters:
    - curves: List of curves, each a list of [x, y] or [x, y, z] coordinates
    - tol: Tolerance for matching points

    Returns:
    - List of ordered coordinates forming one stitched boundary path
    """
    def points_equal(p1, p2):
        return np.allclose(p1, p2, atol=tol)

    curves = curves.copy()
    stitched = curves.pop(0)

    while curves:
        stitched_one = False  # Flag to check if we stitched a curve in this pass

        for i in range(len(curves)):
            curve = curves[i]
            if points_equal(curve[0], stitched[-1]):
                stitched += curve[1:]
            elif points_equal(curve[-1], stitched[-1]):
                stitched += curve[-2::-1]
            elif points_equal(curve[0], stitched[0]):
                stitched = curve[-1:0:-1] + stitched
            elif points_equal(curve[-1], stitched[0]):
                stitched = curve[:-1] + stitched
            else:
                continue  # Curve didn't match, try next one
            curves.pop(i)
            stitched_one = True
            break  # Start over since curves list has changed

        if not stitched_one:
            print("Could not stitch any curve in this pass — possibly disconnected.")
            break

    return stitched

In [151]:
def get_surface_contour(boundary_surfaces):
    list_all_contours = []
    #print("boundary_surfaces: ", boundary_surfaces)
    for boundary_surface in boundary_surfaces:
        #print("boundary_surface: ", boundary_surface)
        list_contours = []
        _, array_curve_tags = gmsh.model.occ.getCurveLoops(boundary_surface)
        #print("array_curve_tags: ", array_curve_tags)
        for array_curve_tag in array_curve_tags:
            list_coords = []
            for curve_tag in array_curve_tag:
                #print("curve tag: ", curve_tag)
                _, coords, parametricCoords = gmsh.model.mesh.getNodes(dim = 1, tag = curve_tag, includeBoundary=True)
                coords = np.reshape(coords, (-1, 3))[:, 0:2] # reshape to (n_points, 3) and  omit z-coordinates
                parametricCoords, coords = remove_duplicate_params(parametricCoords, coords)
                sorted_coords = [x for _, x in sorted(zip(parametricCoords, coords))] # abs(parametricCoords) does weird things... 
                list_coords.append(sorted_coords)
            list_contours.append(list_coords)
        
        #print("total number of contours: ", len(list_contours))

        conc_list_contours = []
        for contour in list_contours:
            #print("nr of segments in contour: ", len(contour))
            #print("contour: ", contour)
            stitched = stitch_curves(contour)
            #print("stitched: ", len(stitched))
            # Flatten all curves into a single list of coordinate arrays
            conc_list_contours.append(stitched)
        list_all_contours.append(conc_list_contours)

    return list_all_contours

In [152]:
def map_to_grid(coords, printing_dimensions, image_size):
    """
    Maps real-world (x, y) coordinates to discrete image grid indices.

    Parameters:
    - coords: tuple or list of (x, y)
    - printing_dimensions: physical dimensions [width, height]
    - image_size: image dimensions [width_px, height_px]

    Returns:
    - (i, j): mapped pixel coordinates (integer indices)
    """
    x, y = coords
    pw, ph = printing_dimensions
    iw, ih = image_size

    # Normalize to range [0, 1]
    u = (x + pw / 2) / pw
    v = (ph / 2 - y) / ph

    # Map to pixel coordinates
    i = int(np.rint(u * iw))
    j = int(np.rint(v * ih))

    return i, j

In [153]:
def polygons_to_mask(list_all_contours, image_size, printing_dimensions):
    """
    Converts a list of 2D polygons into a single 2D binary mask.

    Parameters:
    - polygons: list of polygons, where each polygon is a list of (x, y) tuples.
    - image_size: (height, width) of the output mask.

    Returns:
    - A 2D numpy array of shape (height, width) with 1s inside the polygons and 0s elsewhere.
    """
    mask = np.zeros(image_size, dtype=np.uint8)

    for list_polygons_points in list_all_contours:
        for polygon in list_polygons_points:
            #print("nr of points in connected curve (aka polygon): ", len(polygon))
            mask_ = np.zeros(image_size, dtype=np.uint8)
            pts = []
            for point in polygon:
                x, y = map_to_grid(point, printing_dimensions, image_size)
                pts.append([x, y])
            pts = np.array([pts], dtype=np.int32)
            cv2.fillPoly(mask_, pts, color=True)
            mask = np.logical_xor(mask, mask_)

    return mask

In [154]:
def plot_mask(mask, title="Binary Mask"):
    """
    Plots a 2D binary mask using matplotlib.

    Parameters:
    - mask: 2D numpy array of 0s and 1s.
    - title: Optional title for the plot.
    """
    plt.figure(figsize=(10, 10))
    plt.imshow(mask, cmap='gray', interpolation='nearest')
    plt.title(title)
    plt.axis('off')  # Hide axis ticks/labels
    plt.show()

In [165]:
def generate_layered_meshes(layer_height, layers_lumping, max_element_size, mask_size, buildplate_dimensions, input_file, output_folder):
    if not gmsh.isInitialized():
        gmsh.initialize()
    else: 
        gmsh.clear()
    gmsh.option.setNumber("General.Terminal", 5)

    gmsh.option.setString('Geometry.OCCTargetUnit', 'M')
    initializer = gmsh.model.occ.importShapes(input_file)[0][1] # returns [(dim, tag)] resp only "tag" with [0][1]
    gmsh.model.occ.synchronize()

    xmin, ymin, zmin, xmax, ymax, part_height = gmsh.model.occ.getBoundingBox(3, initializer)
    print("Old bounding box of model: (xmin, ymin, zmin, xmax, ymax, part_height): ", (xmin*1000, ymin*1000, zmin*1000, xmax*1000, ymax*1000, part_height*1000), "[mm]")
    midpoint = np.array([xmin + (xmax - xmin) / 2, ymin + (ymax - ymin) / 2, zmin])
    print("Old Midpoint of the model: ", midpoint)
    gmsh.model.occ.translate([(3, initializer)], - midpoint[0], - midpoint[1], - midpoint[2]) # translate the model to the origin
    gmsh.model.occ.synchronize()
    xmin, ymin, zmin, xmax, ymax, part_height = gmsh.model.occ.getBoundingBox(3, initializer)
    midpoint_new = np.array([xmin + (xmax - xmin) / 2, ymin + (ymax - ymin) / 2, zmin])
    print("New Midpoint of the model: ", midpoint_new)

    n_superlayers = int(part_height / layer_height / layers_lumping)
    superlayer_height = layer_height * layers_lumping
    min_size = superlayer_height / 2

    print("New bounding box of model: (xmin, ymin, zmin, xmax, ymax, part_height): ", (xmin*1000, ymin*1000, zmin*1000, xmax*1000, ymax*1000, part_height*1000), "[mm]")
    print("This many files will be generated: ", n_superlayers)
    
    while True:
        #user_input = input("Continue? (yes/no): ").strip().lower()
        user_input = "yes" # for testing purposes
        if user_input == "yes":
            print("Continuing...")
            break
        elif user_input == "no":
            print("Aborting...")
            return
        else:
            print("Invalid input. Please enter 'yes' or 'no'.")

    gmsh.model.remove()

    os.makedirs(output_folder, exist_ok = True)

    for i in range(1, 2): # range(start, stop [excluding])
    #for i in range(1, n_superlayers + 1): # range(start, stop [excluding])
        gmsh.model.add(f"layer_{i}")
        lz = i * superlayer_height # current height of the superlayer 
        fine_thickness = 3 * superlayer_height # top 3 layers are finer
        fine_start_z = max(0, lz - fine_thickness)
        transition_thickness = 10 * superlayer_height # transition thickness

        cut_step = gmsh.model.occ.importShapes(input_file)[0][1] # returns [(dim, tag)]
        gmsh.model.occ.synchronize()
        print("cut_step: ", cut_step)
        print("before translation: ", gmsh.model.occ.getBoundingBox(3, cut_step))
        # !!! careful: we still need to use the old midpoint for the translation !!! 
        print("will be translated to: ", midpoint)
        gmsh.model.occ.translate([(3, cut_step)], - midpoint[0], - midpoint[1], - midpoint[2]) # translate the model to the origin
        gmsh.model.occ.synchronize()
        print("after translation: ", gmsh.model.occ.getBoundingBox(3, cut_step))
        
        dx = xmax - xmin # side length of box in x-direction
        dy = ymax - ymin # side length of box in y-direction
        dz = lz          # side length of box in z-direction
        print("dz: ", dz)
        cutter = gmsh.model.occ.addBox(xmin, ymin, 0, dx, dy, dz) # returns only the tag of the box (integer)
        gmsh.model.occ.synchronize()
        
        print("cutter bounding box: ", gmsh.model.occ.getBoundingBox(3, cutter))
        print("Cutter tag: ", cutter)
        intersected, _ = gmsh.model.occ.intersect([(3, cut_step)], [(3, cutter)], tag=-1,
                            removeObject=True, removeTool=True)
        print("intersected: ", intersected)
        print("bounding box of intersected: ", gmsh.model.occ.getBoundingBox(3, intersected[0][1]))
        gmsh.model.occ.synchronize()
        
        gmsh.model.occ.removeAllDuplicates()
        gmsh.model.occ.synchronize()

        print("intersected: ", intersected)
        box_id = gmsh.model.mesh.field.add("Box")
        gmsh.model.mesh.field.setNumber(box_id, "VIn", min_size)
        gmsh.model.mesh.field.setNumber(box_id, "VOut", max_element_size)
        gmsh.model.mesh.field.setNumber(box_id, "XMin", xmin)
        gmsh.model.mesh.field.setNumber(box_id, "XMax", xmax)
        gmsh.model.mesh.field.setNumber(box_id, "YMin", ymin)
        gmsh.model.mesh.field.setNumber(box_id, "YMax", ymax)
        gmsh.model.mesh.field.setNumber(box_id, "ZMin", fine_start_z)
        gmsh.model.mesh.field.setNumber(box_id, "ZMax", lz)
        gmsh.model.mesh.field.setNumber(box_id, "Thickness", transition_thickness) # transition thickness 
        gmsh.model.mesh.field.setAsBackgroundMesh(box_id)

        gmsh.option.setNumber("Mesh.MeshSizeMax", max_element_size)
        print("intersected: ", intersected)
        
        volumes = [] 
        for dim, tag in intersected:
            volumes.append(tag)

        print("Number of discrete volumes: ", len(volumes))

        gmsh.model.addPhysicalGroup(3, volumes, tag = 1) 
        gmsh.model.setPhysicalName(3, 1, f"volume_of_superlayer_{i}")

        surfaces = gmsh.model.getBoundary(intersected, oriented=False, recursive=False)
        top, bottom, sides = [], [], []
        for dim, tag in surfaces:
            _, _, com_z = gmsh.model.occ.getCenterOfMass(dim, tag) # returns the center of mass of each surface: (x, y, z) where we only care about z
            print("com_z: ", com_z*1000, "[mm]")
            if abs(com_z - lz) < 1e-6:
                top.append(tag)
                gmsh.model.geo.mesh.setTransfiniteSurface(tag)
            elif abs(com_z) < 1e-6:
                bottom.append(tag)
                gmsh.model.geo.mesh.setTransfiniteSurface(tag)
            else:
                sides.append(tag)

        gmsh.model.geo.synchronize()

        gmsh.model.addPhysicalGroup(2, top, tag=13)
        gmsh.model.setPhysicalName(2, 13, "top")

        gmsh.model.addPhysicalGroup(2, bottom, tag=14)
        gmsh.model.setPhysicalName(2, 14, "bottom")

        gmsh.model.addPhysicalGroup(2, sides, tag=15)
        gmsh.model.setPhysicalName(2, 15, "sides")

        gmsh.model.mesh.generate(2)

        boundary_surfaces = gmsh.model.getEntitiesForPhysicalGroup(dim = 2, tag = 13) # [2 9]

        # "list_contours" has shape "nr of surfaces" x "nr of curves of surface" x "nr of points on curve" x "dimension (x, y)"
        list_all_contours = get_surface_contour(boundary_surfaces) 
        print("list_all_contours: ", list_all_contours)

        # Now generate a mask
        mask = polygons_to_mask(list_all_contours, mask_size, buildplate_dimensions)
        mask_path = os.path.join(output_folder, f"mask_{i:05d}")
        np.save(mask_path, mask)
        plot_mask(mask, title=f"Superlayer {i} Mask, height: {lz*1000} mm")

        gmsh.model.occ.removeAllDuplicates()
        gmsh.model.occ.synchronize()

        gmsh.model.mesh.generate(3)
        gmsh.model.mesh.removeDuplicateNodes()

        #gmsh.model.mesh.optimize("Netgen")

        mesh_path = os.path.join(output_folder, f"superlayer_{i:05d}.msh")
        gmsh.write(mesh_path)
        gmsh.model.remove()

    gmsh.finalize()


# ---- Run it ----
if __name__ == "__main__":
    layer_height = 50 * 1e-6 # converts um to m 
    layers_lumping = 10
    max_element_size = 1 * 1e-3 # converts mm to m

    mask_size = (100, 100) # in ([px], [px])
    buildplate_dimensions = (200, 200) # in ([mm], [mm]) 

    input_file="bunny.step"
    output_folder="bunny_superlayers_10_lumped"

    generate_layered_meshes(
        layer_height, 
        layers_lumping = layers_lumping, 
        max_element_size = max_element_size, 
        mask_size = mask_size,
        buildplate_dimensions = buildplate_dimensions,
        input_file=input_file, 
        output_folder=output_folder)
    
    print("Mesh generation completed.")

Info    : Clearing all models and views...
Info    : Done clearing all models and views
Info    :  - Label 'Shapes/bunny v2' (3D)
Info    :  - Color (0.627451, 0.627451, 0.627451) (3D & Surfaces)
Old bounding box of model: (xmin, ymin, zmin, xmax, ymax, part_height):  (-86.35284211956967, -66.94995550374968, 0.6219123589161264, 86.37478943864967, 66.55894822748968, 170.85182854770966) [mm]
Old Midpoint of the model:  [ 1.09736595e-05 -1.95503638e-04  6.21912359e-04]
New Midpoint of the model:  [0.00000000e+00 0.00000000e+00 1.86347248e-20]
New bounding box of model: (xmin, ymin, zmin, xmax, ymax, part_height):  (-86.36381577910967, -66.75445186561969, 1.8634724839594607e-17, 86.36381577910966, 66.75445186561967, 170.22991618879354) [mm]
This many files will be generated:  340
Continuing...
Info    :  - Label 'Shapes/bunny v2' (3D)
Info    :  - Color (0.627451, 0.627451, 0.627451) (3D & Surfaces)
cut_step:  1
before translation:  (-0.08635284211956967, -0.06694995550374967, 0.0006219123

IndexError: list index out of range

In [ ]:
def load_first_layer(msh_file):
    # Convert mesh if needed and import
    print(type(msh_file))
    mesh_, cell_tags, facet_tags = gmshio.read_from_msh(msh_file, comm = MPI.COMM_WORLD, rank=0, gdim=3)
    return mesh_, cell_tags, facet_tags


def plot_dolfinx_mesh(mesh_, cell_type=3):
    """
    Plot a dolfinx mesh using PyVista.
    
    Parameters:
    - mesh: dolfinx.cpp.mesh.Mesh
    - cell_type: 2 for triangle (2D), 3 for tetrahedron (3D)
    """
    grid = pv.UnstructuredGrid(*vtk_mesh(mesh_, mesh_.topology.dim))

    plotter = pv.Plotter()
    plotter.add_mesh(grid, show_edges=True, color="lightblue", opacity=0.7)
    plotter.show()

In [7]:
%%px

def write_partitioned_mesh(filename: Path):
    import subprocess
    from mpi4py import MPI
    import dolfinx
    import adios4dolfinx

    mesh, cell_tags, facet_tags = load_first_layer(os.path.join("layers", "layer_42.msh"))
    
    # Write mesh checkpoint
    adios4dolfinx.write_mesh(filename, mesh, engine="BP4", store_partition_info=True)
    adios4dolfinx.write_meshtags(filename, mesh, cell_tags, engine="BP4", meshtag_name = "cells")
    adios4dolfinx.write_meshtags(filename, mesh, facet_tags, engine="BP4", meshtag_name = "facets")
    # Inspect checkpoint on rank 0 with `bpls`
    if mesh.comm.rank == 0:
        output = subprocess.run(["bpls", "-a", "-l", filename], capture_output=True)
        print(output.stdout.decode("utf-8"))


def read_partitioned_mesh(filename: Path, read_from_partition: bool = True):
    from mpi4py import MPI

    import adios4dolfinx

    prefix = f"{MPI.COMM_WORLD.rank + 1}/{MPI.COMM_WORLD.size}: "
    try:
        mesh = adios4dolfinx.read_mesh(
            filename, comm=MPI.COMM_WORLD, engine="BP4", read_from_partition=read_from_partition
        )
        cell_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "cells", engine="BP4")
        facet_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "facets", engine="BP4")

        tdim = mesh.topology.dim
        mesh.topology.create_connectivity(tdim - 1, tdim)

        print(f"{prefix} Mesh: {mesh.name} read successfully with {read_from_partition=}")
    except ValueError as e:
        print(f"{prefix} Caught exception: ", e)

    return mesh, cell_tags, facet_tags

In [8]:
%%px 

import time 

mesh_file = Path("partitioned_mesh.bp")
#write_partitioned_mesh(mesh_file)

start_time = time.time()
mesh2, cell_tags2, facet_tags2 = read_partitioned_mesh(mesh_file, True)
end_time = time.time()
print(f"Time taken to read mesh: {(end_time - start_time)*1000} ms")
print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")
#print(len(mesh.cell_tags.values), "DOFs for this process")

[stdout:0] 1/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1201.0531425476074 ms
Loaded mesh with 128724 cells.


[stdout:4] 5/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1219.501256942749 ms
Loaded mesh with 129263 cells.


[stdout:6] 7/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1223.1879234313965 ms
Loaded mesh with 129157 cells.


[stdout:2] 3/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1224.984884262085 ms
Loaded mesh with 129647 cells.


[stdout:7] 8/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1225.876808166504 ms
Loaded mesh with 129193 cells.


[stdout:1] 2/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1231.5773963928223 ms
Loaded mesh with 129920 cells.


[stdout:9] 10/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1237.7893924713135 ms
Loaded mesh with 129172 cells.


[stdout:3] 4/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1246.0906505584717 ms
Loaded mesh with 129006 cells.


[stdout:8] 9/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1245.710849761963 ms
Loaded mesh with 129190 cells.


[stdout:5] 6/10:  Mesh: mesh read successfully with read_from_partition=True
Time taken to read mesh: 1250.774621963501 ms
Loaded mesh with 129159 cells.


In [ ]:
%%px 

import cProfile, pstats
profiler = cProfile.Profile()
profiler.enable()
_, _, _ = _(os.path.join("layers", "layer_42.msh")) # /mnt/ramdisk/ slower than layers
profiler.disable()
stats = pstats.Stats(profiler).sort_stats('ncalls')
stats.print_stats()

[stdout:0] <class 'str'>
Info    : Reading 'layers/layer_42.msh'...
Info    : 27 entities
Info    : 234666 nodes
Info    : 1401499 elements
Info    : Done reading 'layers/layer_42.msh'
         10781 function calls (10046 primitive calls) in 6.488 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
1105/1077    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
      894    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      639    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
   412/68    0.000    0.000    0.005    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      322    0.000    0.000    0.000

[stdout:1] <class 'str'>
         6691 function calls (6024 primitive calls) in 6.452 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      753    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      604    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  500/492    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.004    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      303    0.000    0.000 

[stdout:5] <class 'str'>
         6827 function calls (6178 primitive calls) in 6.416 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      755    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      605    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  543/535    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.003    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      304    0.000    0.000 

[stdout:6] <class 'str'>
         6875 function calls (6210 primitive calls) in 6.429 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      762    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      605    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  544/536    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.002    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      304    0.000    0.000 

[stdout:7] <class 'str'>
         6875 function calls (6210 primitive calls) in 6.453 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      762    0.000    0.000    0.000    0.000 {built-in method builtins.len}
      656    0.000    0.000    0.000    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:332(_ufl_hash_data_)
      605    0.000    0.000    0.000    0.000 {method 'append' of 'list' objects}
      600    0.000    0.000    0.000    0.000 {built-in method _weakref.proxy}
  544/536    0.000    0.000    0.001    0.000 {built-in method builtins.isinstance}
   412/68    0.000    0.000    0.002    0.000 /home/dan/miniconda3/envs/fenicsx-env/lib/python3.13/site-packages/ufl/cell.py:269(<genexpr>)
      306    0.000    0.000    0.000    0.000 {built-in method _abc._abc_subclasscheck}
      306    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
      304    0.000    0.000 

...

Out[4:10]: <pstats.Stats at 0x7f802f207d90>

Out[2:10]: <pstats.Stats at 0x7f6d5129fd90>

Out[7:10]: <pstats.Stats at 0x7f2458667d90>

Out[3:10]: <pstats.Stats at 0x7fe49cb2bd90>

Out[0:10]: <pstats.Stats at 0x7fc7db613d90>

In [69]:
%%px 

mesh1, cell_tags1, facet_tags1 = load_first_layer(os.path.join("layers", "layer_66.msh"))

print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")

#plot_dolfinx_mesh(mesh1)


[stdout:0] <class 'str'>
Info    : Reading 'layers/layer_66.msh'...
Info    : 27 entities
Info    : 183760 nodes
Info    : 1122936 elements
Info    : Done reading 'layers/layer_66.msh'
Loaded mesh with 373908 cells.


[stdout:1] <class 'str'>
Loaded mesh with 373778 cells.


[stdout:2] <class 'str'>
Loaded mesh with 373784 cells.


%px:   0%|          | 0/3 [00:00<?, ?tasks/s]